# Socioeconomic distributions

Default paths are relative: inputs under `data/`, results under `outputs/06_socioeconomic/`. Use the source cached country-income mapping and predicted countries by default; no external requests are made. Optional collection writes new files to the configured results directory without replacing historical inputs. A first ROR search result is an unverified affiliation-country candidate, not a validated institutional match. Surname-origin predictions are not verified nationality or residence.

Use one resolved country per author ID. Agreeing repeats collapse; conflicting predictions remain missing. Country-income mappings with conflicting groups remain unavailable rather than being arbitrarily selected. The output reports these exclusions. Income proportions condition on a resolved country with an unambiguous income mapping; top-country proportions condition on a resolved country. No author-level data rows or identifying text are displayed.

In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
OUTPUT_DIR = REPO_ROOT / 'outputs' / '06_socioeconomic'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
import pandas as pd
RUN_DATA_COLLECTION = False

def load_country_predictions(path,column):
    raw = pd.read_csv(path,usecols=['id',column],encoding='utf-8')
    if raw.id.isna().any():
        raise ValueError('Author IDs must be nonmissing.')
    raw[column] = raw[column].replace(['','None','NULL'],pd.NA)
    conflicts = raw.groupby('id')[column].nunique().gt(1)
    resolved = raw.groupby('id',sort=False)[column].first()
    resolved.loc[conflicts[conflicts].index] = pd.NA
    print('Country prediction audit:', 'input rows',len(raw),'unique authors',len(resolved),'conflicting authors',int(conflicts.sum()),'unresolved authors',int(resolved.isna().sum()))
    return resolved.rename(column).reset_index()

income_raw = pd.read_csv(DATA_DIR/'country_income.csv',usecols=['Country','IncomeGroup'])
income_conflicts = income_raw.groupby('Country').IncomeGroup.nunique().gt(1)
income_df = income_raw[~income_raw.Country.isin(income_conflicts[income_conflicts].index)].dropna().drop_duplicates('Country')
assert income_df.Country.is_unique
print('Country-income audit:', 'input rows',len(income_raw),'conflicting countries excluded',int(income_conflicts.sum()))


Call Worlds Bank API to get Country Income Classification

In [ ]:
if RUN_DATA_COLLECTION:
    import requests
    import pandas as pd

    # World Bank API endpoints for each income level
    API_URL = "https://api.worldbank.org/v2/country"
    INCOME_CODES = {
        'Low income': 'LIC',
        'Lower middle income': 'LMC',
        'Upper middle income': 'UMC',
        'High income': 'HIC'
    }

    records = []
    for tier, code in INCOME_CODES.items():
        # fetch all pages
        page = 1
        while True:
            r = requests.get(API_URL,
                             params={'incomeLevel': code, 'format': 'json', 'page': page, 'per_page': 1000}, timeout=30)
            r.raise_for_status()
            data = r.json()
            # data[1] is the list of countries
            for c in data[1]:
                records.append({
                    'Country': c['name'],
                    'IncomeGroup': tier
                })
            # pagination
            total = int(data[0]['total'])
            per_page = int(data[0]['per_page'])
            if page * per_page >= total:
                break
            page += 1

    # save to CSV
    df = pd.DataFrame(records)
    df.to_csv(str(OUTPUT_DIR/'country_income_download.csv'), index=False, encoding='utf-8')
    print("Saved country_income.csv with", len(df), "rows.")


Call ROR API to get the country of affiliation

In [ ]:
if RUN_DATA_COLLECTION:
    import pandas as pd
    import requests
    import time
    import os
    import csv

    INPUT_FILE = str(DATA_DIR / 'pred/dataset/csv/authors.csv')
    OUTPUT_FILE = str(OUTPUT_DIR/'pred_authors_country_affiliation_download.csv')


    def get_country_from_ror(aff):
        if pd.isna(aff) or aff == "NULL" or aff.strip() == "":
            return None
        try:
            url = "https://api.ror.org/organizations"
            resp = requests.get(url, params={"query": aff}, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                if data.get("number_of_results", 0) > 0:
                    first_item = data["items"][0]
                    if "locations" in first_item and first_item["locations"]:
                        return first_item["locations"][0]["geonames_details"].get("country_name")
        except Exception as e:
            print("Affiliation lookup failed; no identifying text is logged.")
        return None


    def main():
        df = pd.read_csv(INPUT_FILE)

        df = df[
       
            df["affiliations"].notna()
            & (df["affiliations"] != "NULL")
            & (df["affiliations"].str.strip() != "")
        ].copy()

        print(f"There are {len(df)} records to process.")

        if os.path.exists(OUTPUT_FILE) and os.path.getsize(OUTPUT_FILE) > 0:
            done_df = pd.read_csv(OUTPUT_FILE)
            done_ids = set(done_df["id"].astype(str))
            print(f"{len(done_ids)} results detected, these will be skipped.")
            write_header = False
        else:
            done_ids = set()
            write_header = True

        with open(OUTPUT_FILE, "a", encoding="utf-8", newline="") as f_out:
            writer = csv.writer(f_out)

            if write_header:
                writer.writerow(["id", "name", "gs_a_id", "affiliations", "country"])

            for _, row in df.iterrows():
                author_id = str(row["id"])
                if author_id in done_ids:
                    continue 

                aff = row["affiliations"]
                country = get_country_from_ror(aff)

                writer.writerow([author_id, row["name"], row["gs_a_id"], aff, country])
                f_out.flush()

                time.sleep(1)  


    if __name__ == "__main__":
        main()


Income Class Distribution - affiliation based

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# affiliation based
bad_df  = load_country_predictions(str(DATA_DIR / 'pred/metadata/pred_authors_country_affiliation.csv'),'country')
good_df = load_country_predictions(str(DATA_DIR / 'norm/metadata/norm_authors_country_affiliation.csv'),'country')
# income_df is the validated cached mapping from configuration.  

bad_map  = pd.merge(
    bad_df[['country']].rename(columns={'country':'Country'}),
    income_df,
    on='Country',
    how='inner', validate='many_to_one'
)
good_map = pd.merge(
    good_df[['country']].rename(columns={'country':'Country'}),
    income_df,
    on='Country',
    how='inner', validate='many_to_one'
)

bad_counts  = bad_map ['IncomeGroup'].value_counts().sort_index()
good_counts = good_map['IncomeGroup'].value_counts().sort_index()

contingency = pd.DataFrame({
    'Predatory': bad_counts,
    'Normal'  : good_counts
}).fillna(0).astype(int)

chi2, p, dof, expected = chi2_contingency(contingency)

proportions = contingency.div(contingency.sum(axis=0), axis=1)

print("Contingency Table (counts):")
print(contingency, "\n")

print("Proportions by IncomeGroup:")
print(proportions, "\n")

print(f"Chi² = {chi2:.3f}, p = {p:.3g}, dof = {dof}\n")

print("Expected frequencies under H0 (no difference):")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns))


Income Class Distribution - lastname based

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# name based
bad_df  = load_country_predictions(str(DATA_DIR / 'pred/metadata/pred_author_country_lastname.csv'),'Origin Country')
good_df = load_country_predictions(str(DATA_DIR / 'norm/metadata/norm_author_country_lastname.csv'),'Origin Country')
# income_df is the validated cached mapping from configuration. 

bad_map  = pd.merge(
    bad_df[['Origin Country']].rename(columns={'Origin Country':'Country'}),
    income_df,
    on='Country',
    how='inner', validate='many_to_one'
)
good_map = pd.merge(
    good_df[['Origin Country']].rename(columns={'Origin Country':'Country'}),
    income_df,
    on='Country',
    how='inner', validate='many_to_one'
)

bad_counts  = bad_map ['IncomeGroup'].value_counts().sort_index()
good_counts = good_map['IncomeGroup'].value_counts().sort_index()

contingency = pd.DataFrame({
    'Predatory': bad_counts,
    'Normal'  : good_counts
}).fillna(0).astype(int)

chi2, p, dof, expected = chi2_contingency(contingency)

proportions = contingency.div(contingency.sum(axis=0), axis=1)

print("Contingency Table (counts):")
print(contingency, "\n")

print("Proportions by IncomeGroup:")
print(proportions, "\n")

print(f"Chi² = {chi2:.3f}, p = {p:.3g}, dof = {dof}\n")

print("Expected frequencies under H0 (no difference):")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns))


Top 10 Countries Distribution

In [ ]:
import pandas as pd

def top_affiliation_countries(file_path, top_n=10, save_path=None):

    df = load_country_predictions(file_path,'country')

    df = df[df["country"].notna() & (df["country"] != "None")]

    counts = df["country"].value_counts()

    top = counts.head(top_n)

    total = len(df)
    percentages = (top / total * 100).round(2)

    result = pd.DataFrame({
        "count": top,
        "percentage": percentages
    })

    if save_path:
        result.to_csv(save_path)

    return result

def top_lastname_countries(file_path, top_n=10, save_path=None):
    df = load_country_predictions(file_path,'Origin Country')
    df = df[df["Origin Country"].notna() & (df["Origin Country"] != "None")]
    counts = df["Origin Country"].value_counts()
    top = counts.head(top_n)
    total = len(df)
    percentages = (top / total * 100).round(2)

    result = pd.DataFrame({
        "count": top,
        "percentage": percentages
    })

    if save_path:
        result.to_csv(save_path)

    return result



In [ ]:
res = top_affiliation_countries(str(DATA_DIR / 'norm/metadata/norm_authors_country_affiliation.csv'), top_n=10)
res

In [ ]:
res = top_affiliation_countries(str(DATA_DIR / 'pred/metadata/pred_authors_country_affiliation.csv'), top_n=10)
res

In [ ]:
res = top_lastname_countries(str(DATA_DIR / 'norm/metadata/norm_author_country_lastname.csv'), top_n=10)
res

In [ ]:
res = top_lastname_countries(str(DATA_DIR / 'pred/metadata/pred_author_country_lastname.csv'), top_n=10)
res

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def compare_country_distributions_ab(
    file1_a, file2_a, file1_b, file2_b,
    column_a="country", column_b="Origin Country",
    top_n=10, 
    labels=("Dataset1", "Dataset2"), 
    save_path=None
):
    plt.rcParams.update({'font.size': 13})

    COLOR_Normal = '#3e99bb'
    COLOR_Pred = '#a3245d'
    
    def load_counts(path, column_name):
        df = load_country_predictions(path,column_name)
        df = df[df[column_name].notna() & (df[column_name] != "None")]
        counts = df[column_name].value_counts(normalize=False)
        return counts, len(df)

    def prepare_data(file1, file2, column_name):
        counts1, total1 = load_counts(file1, column_name)
        counts2, total2 = load_counts(file2, column_name)
        
        combined = (counts1.add(counts2, fill_value=0)).sort_values(ascending=False)
        top_countries = combined.head(top_n).index.tolist()
        
        data = pd.DataFrame({
            labels[0]: counts1.reindex(top_countries).fillna(0).astype(int),
            labels[1]: counts2.reindex(top_countries).fillna(0).astype(int),
        }, index=top_countries)
        
        data_pct = pd.DataFrame({
            labels[0]: (data[labels[0]] / total1 * 100).round(2),
            labels[1]: (data[labels[1]] / total2 * 100).round(2),
        }, index=top_countries)
        
        return data_pct

    data_pct_a = prepare_data(file1_a, file2_a, column_a)
    data_pct_b = prepare_data(file1_b, file2_b, column_b)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

    data_pct_a.plot(
        kind="bar", 
        ax=ax1, 
        width=0.8,
        color=[COLOR_Pred, COLOR_Normal],
        edgecolor='black',
        linewidth=1,
        legend=False
    )
 
    bars = ax1.patches
    n_countries = len(data_pct_a)
    for i, bar in enumerate(bars):
        if i >= n_countries: 
            bar.set_hatch('\\\\')
    
    ax1.set_ylabel("Percentage (%)", fontsize=20, labelpad=10)
    ax1.tick_params(axis='x', rotation=45, labelsize=17)
    ax1.tick_params(axis='y', labelsize=18)
    ax1.set_xticklabels(ax1.get_xticklabels(), ha="right")
    ax1.text(0, 1.09, 'A', transform=ax1.transAxes, 
             fontsize=23, fontweight='bold', va='top')

    data_pct_b.plot(
        kind="bar", 
        ax=ax2, 
        width=0.8,
        color=[COLOR_Pred, COLOR_Normal],
        edgecolor='black',
        linewidth=1,
        legend=False
    )
   
    bars = ax2.patches
    n_countries = len(data_pct_b)
    for i, bar in enumerate(bars):
        if i >= n_countries: 
            bar.set_hatch('\\\\')
    
    ax2.tick_params(axis='x', rotation=45, labelsize=17)
    ax2.tick_params(axis='y', labelleft=False)
    ax2.set_xticklabels(ax2.get_xticklabels(), ha="right")
    ax2.text(0, 1.09, 'B', transform=ax2.transAxes, 
             fontsize=23, fontweight='bold', va='top')

    handles, legend_labels = ax1.get_legend_handles_labels()
    fig.legend(
        handles, legend_labels,
        loc='upper center', bbox_to_anchor=(0.5, 1.0),
        ncol=2, fontsize=23, title_fontsize=18, frameon=False
    )

    plt.tight_layout(rect=[0, 0, 1, 0.92])

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    return data_pct_a, data_pct_b


res_a, res_b = compare_country_distributions_ab(
    file1_a=str(DATA_DIR / 'pred/metadata/pred_authors_country_affiliation.csv'),
    file2_a=str(DATA_DIR / 'norm/metadata/norm_authors_country_affiliation.csv'),
    file1_b=str(DATA_DIR / 'pred/metadata/pred_author_country_lastname.csv'),
    file2_b=str(DATA_DIR / 'norm/metadata/norm_author_country_lastname.csv'),
    column_a="country",
    column_b="Origin Country",
    top_n=10,
    labels=("Predatory", "Normal"),
    save_path=str(OUTPUT_DIR / 'country_comparison_ab.png')
)

print("Plot A results:")
print(res_a)
print("\nPlot B results:")
print(res_b)